Once you have isolated VNets, you need ways to connect them — to each other and to on-premises networks. Azure provides three primary mechanisms:

- **VNet Peering** — direct, low-latency connection between two VNets over the Azure backbone
- **VPN Gateway** — encrypted IPsec/IKE tunnel over the public internet (or Microsoft backbone) to on-premises or other VNets
- **ExpressRoute** — private, dedicated circuit from your on-premises network to Azure, bypassing the internet entirely

Each option has different trade-offs in cost, latency, bandwidth, and complexity. Understanding when to use each is a core AZ-104 and real-world architecture skill.

## VNet Peering

**VNet Peering** connects two VNets so their resources can communicate privately using private IP addresses over the Azure backbone — with low latency and high bandwidth, equivalent to resources in the same VNet.

### Types of peering

| Type | Scope |
|---|---|
| **Regional peering** | Two VNets in the same Azure region |
| **Global peering** | Two VNets in different Azure regions |

Both types are functionally identical. Global peering has a slightly higher data transfer cost.

### Key properties

- **Non-transitive** — peering A↔B and B↔C does NOT give A access to C. You must peer A↔C explicitly, or use a hub-and-spoke topology with a gateway or Azure Firewall handling transit.
- **Bidirectional but separately configured** — peering is created in both VNets; each side has its own settings
- **No overlapping address spaces** — VNets being peered must have non-overlapping CIDR blocks
- **Works across subscriptions and Entra ID tenants** — useful for large enterprise topologies

### Peering settings

| Setting | Description |
|---|---|
| **Allow gateway transit** | Let the peered VNet use this VNet's VPN/ExpressRoute gateway |
| **Use remote gateways** | Use the gateway in the peered VNet to reach on-premises |
| **Allow forwarded traffic** | Accept traffic that did not originate in the peered VNet (required for hub-and-spoke) |
| **Allow virtual network access** | Enable/disable traffic between the two VNets (default: enabled) |

### Hub-and-spoke topology

The most common enterprise VNet design:

```
          Spoke A (10.1.0.0/16)
              ↕  peering
On-prem  ←→  Hub VNet (10.0.0.0/16)  ←→  Spoke B (10.2.0.0/16)
  (VPN/ER)       ↕ Azure Firewall            ↕  peering
              Spoke C (10.3.0.0/16)
```

- **Hub** — contains shared services: VPN/ExpressRoute gateway, Azure Firewall, Bastion, DNS
- **Spokes** — individual workload VNets peered to the hub
- Traffic between spokes flows through the hub (firewall handles inspection)
- Enable **gateway transit** on hub peering + **use remote gateways** on spoke peering so spokes reach on-premises via the hub's gateway

## VPN Gateway

**Azure VPN Gateway** establishes encrypted IPsec/IKEv2 tunnels between your Azure VNet and on-premises networks or other VNets. The gateway is deployed into a dedicated subnet named **`GatewaySubnet`** (minimum `/27`).

### VPN Gateway SKUs

| SKU | Max throughput | Max S2S tunnels | Zone-redundant |
|---|---|---|---|
| **Basic** | 100 Mbps | 10 | No |
| **VpnGw1 / VpnGw1AZ** | 650 Mbps | 30 | AZ version only |
| **VpnGw2 / VpnGw2AZ** | 1 Gbps | 30 | AZ version only |
| **VpnGw3 / VpnGw3AZ** | 1.25 Gbps | 30 | AZ version only |
| **VpnGw4 / VpnGw4AZ** | 5 Gbps | 100 | AZ version only |
| **VpnGw5 / VpnGw5AZ** | 10 Gbps | 100 | AZ version only |

> Use `AZ` SKUs (e.g. `VpnGw1AZ`) for zone-redundant gateways — they survive an AZ failure. Basic SKU does not support active-active mode, BGP, or resize.

### Connection types

| Type | Description |
|---|---|
| **Site-to-Site (S2S)** | Persistent IPsec tunnel from an on-premises VPN device to Azure VPN Gateway |
| **Point-to-Site (P2S)** | Individual client device (laptop) VPN connection to the Azure VNet |
| **VNet-to-VNet** | IPsec tunnel between two Azure VPN Gateways (use peering instead when possible) |

### Site-to-Site VPN

```
On-premises VPN device  ←→  Azure VPN Gateway
  (public IP required)         (GatewaySubnet)
```

- Requires a **Local Network Gateway** resource in Azure — defines the on-premises IP address and address space
- Supports **BGP** for dynamic route exchange — useful when on-premises routes change frequently
- **Active-active mode** deploys two gateway instances for redundancy; both instances maintain tunnels to the on-premises device

### Point-to-Site VPN

P2S lets individual devices connect to the VNet without a VPN appliance. Supported protocols:

| Protocol | Use case |
|---|---|
| **OpenVPN (SSL/TLS)** | Cross-platform: Windows, Linux, macOS, iOS, Android |
| **IKEv2** | Windows and macOS native VPN clients |
| **SSTP** | Windows only; uses HTTPS port 443 — traverses most firewalls |

Authentication options: Azure certificate, Entra ID (OAuth 2.0), RADIUS server.

### VPN Gateway high availability

| Mode | Description |
|---|---|
| **Active-standby** (default) | One active instance; standby takes over in ~10–90 s on failure |
| **Active-active** | Both instances active; each maintains a tunnel to on-premises; faster failover |

## ExpressRoute

**Azure ExpressRoute** provides a **private, dedicated network connection** from your on-premises infrastructure to Azure — bypassing the public internet entirely. The connection is provisioned through an **ExpressRoute partner** (a network service provider like AT&T, Equinix, or Verizon) or directly via **ExpressRoute Direct**.

```
On-premises  →  Partner edge (colocation)  →  Microsoft edge  →  Azure region
                                               (private peering)
```

### ExpressRoute circuit SKUs

| SKU | Reach |
|---|---|
| **Local** | Connect to a single Azure region near the peering location; unlimited outbound data |
| **Standard** | Connect to regions in the same geopolitical region |
| **Premium** | Connect to any Azure region globally; required for Microsoft 365 and Global Reach |

### Bandwidth options

50 Mbps, 100 Mbps, 200 Mbps, 500 Mbps, 1 Gbps, 2 Gbps, 5 Gbps, 10 Gbps — provisioned by the partner.

**ExpressRoute Direct** offers 10 Gbps or 100 Gbps ports connected directly to the Microsoft global network — for customers with very high bandwidth requirements.

### Peering types (BGP sessions)

| Peering type | Traffic |
|---|---|
| **Azure private peering** | Azure VNets — your private resources |
| **Microsoft peering** | Microsoft 365, Dynamics 365, Azure PaaS public endpoints |

> Public peering was retired; use Microsoft peering for Azure public services.

### ExpressRoute Global Reach

**Global Reach** links two ExpressRoute circuits together through the Microsoft backbone — allowing two on-premises sites to communicate via Microsoft's network without traffic touching the public internet.

```
Site A  →  ER circuit A  →  Microsoft backbone  ←  ER circuit B  ←  Site B
```

### ExpressRoute FastPath

By default, ExpressRoute traffic flows through the ExpressRoute Gateway — adding a small hop. **FastPath** bypasses the gateway for data-plane traffic, sending it directly to VNet resources. This improves throughput and reduces latency for the highest-bandwidth circuits (Ultra Performance and ErGw3AZ gateway SKUs).

### ExpressRoute high availability

- Each circuit has **two connections** (primary and secondary) from the provider edge to two Microsoft edge routers — no single point of failure at the edge
- For full redundancy, deploy **two circuits in different peering locations** and connect both to your ExpressRoute Gateway
- For zone redundancy in Azure, use the **ErGwAZ** gateway SKU

## ExpressRoute vs VPN Gateway

| | VPN Gateway | ExpressRoute |
|---|---|---|
| **Connectivity** | Over public internet (encrypted) | Private dedicated circuit |
| **Max bandwidth** | 10 Gbps (VpnGw5) | 100 Gbps (ExpressRoute Direct) |
| **Latency** | Variable (internet) | Consistent, low (private) |
| **SLA** | 99.95% (active-active) | 99.95% |
| **Setup time** | Minutes to hours | Weeks (partner provisioning) |
| **Cost** | Lower | Higher |
| **Use case** | Branch offices, remote access, dev/test | Enterprise, compliance, high bandwidth |

### Coexisting VPN and ExpressRoute

You can deploy **both** simultaneously on the same VNet:
- ExpressRoute as the primary path — lower latency, higher bandwidth
- VPN as a backup — if ExpressRoute fails, traffic fails over to the VPN tunnel

Requires separate gateways in the `GatewaySubnet` — one ExpressRoute Gateway and one VPN Gateway — or a combined **ExpressRoute + VPN coexist** configuration using a single gateway subnet.

## Azure Virtual WAN

**Azure Virtual WAN (vWAN)** is a managed networking service that simplifies hub-and-spoke at scale. Instead of manually configuring peerings and gateways, you create a **Virtual Hub** — a Microsoft-managed hub VNet — and connect:

- Branch offices via S2S VPN
- Remote users via P2S VPN
- On-premises data centres via ExpressRoute
- Spoke VNets via hub connections (managed peering)

Azure handles all the route propagation, gateway management, and transit connectivity between branches and spokes.

| | Manual hub-and-spoke | Azure Virtual WAN |
|---|---|---|
| Hub management | You manage the hub VNet | Microsoft-managed Virtual Hub |
| Transit routing | Requires UDRs + NVA | Built-in |
| Any-to-any connectivity | Manual | Automatic |
| Scale | Limited | Up to thousands of branches |

### vWAN SKUs

| SKU | Capabilities |
|---|---|
| **Basic** | S2S VPN only |
| **Standard** | S2S VPN, P2S VPN, ExpressRoute, VNet connections, inter-hub transit |

## Connectivity Decision Guide

| Scenario | Recommended option |
|---|---|
| Connect two Azure VNets in the same or different regions | **VNet Peering** |
| Small branch office to Azure (< 1 Gbps, internet is fine) | **VPN Gateway S2S** |
| Developer laptop access to Azure VNet | **VPN Gateway P2S** |
| Enterprise data centre — compliance, consistent latency | **ExpressRoute** |
| Maximum bandwidth (10–100 Gbps) | **ExpressRoute Direct** |
| Two on-premises sites via Microsoft backbone | **ExpressRoute Global Reach** |
| ExpressRoute + VPN failover | **Coexisting gateways** |
| Many branches + spokes at scale | **Azure Virtual WAN** |
| Spoke-to-spoke routing without direct peering | **Hub VNet with Azure Firewall + UDRs** |

## Working with Peering and VPN Gateway via Python

In [ ]:
# pip install azure-identity azure-mgmt-network

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.network import NetworkManagementClient
from azure.mgmt.network.models import (
    VirtualNetworkPeering,
    VirtualNetworkGateway, VirtualNetworkGatewaySku,
    VirtualNetworkGatewayIPConfiguration, SubResource,
    LocalNetworkGateway, AddressSpace,
    VirtualNetworkGatewayConnection
)

credential      = DefaultAzureCredential()
subscription_id = "<your-subscription-id>"
resource_group  = "my-rg"
location        = "eastus"

net = NetworkManagementClient(credential, subscription_id)

In [ ]:
# Peer hub-vnet → spoke-vnet  (must also create the reverse peering spoke → hub)
hub_vnet_id   = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Network/virtualNetworks/hub-vnet"
spoke_vnet_id = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Network/virtualNetworks/spoke-vnet"

# Hub side — enable gateway transit so spokes can use the hub's gateway
hub_peering_params = VirtualNetworkPeering(
    allow_virtual_network_access=True,
    allow_forwarded_traffic=True,
    allow_gateway_transit=True,      # hub has the gateway
    use_remote_gateways=False,
    remote_virtual_network=SubResource(id=spoke_vnet_id)
)
poller = net.virtual_network_peerings.begin_create_or_update(
    resource_group, "hub-vnet", "hub-to-spoke", hub_peering_params
)
print(f"Hub peering state: {poller.result().peering_state}")

# Spoke side — use remote gateways so spoke reaches on-premises via hub gateway
spoke_peering_params = VirtualNetworkPeering(
    allow_virtual_network_access=True,
    allow_forwarded_traffic=True,
    allow_gateway_transit=False,
    use_remote_gateways=True,        # use the hub's gateway
    remote_virtual_network=SubResource(id=hub_vnet_id)
)
poller = net.virtual_network_peerings.begin_create_or_update(
    resource_group, "spoke-vnet", "spoke-to-hub", spoke_peering_params
)
print(f"Spoke peering state: {poller.result().peering_state}")

In [ ]:
# List all peerings for a VNet
for p in net.virtual_network_peerings.list(resource_group, "hub-vnet"):
    remote_name = p.remote_virtual_network.id.split("/")[-1]
    print(f"  {p.name:<30} → {remote_name:<20} state: {p.peering_state}")

In [ ]:
# Create a VPN Gateway in the GatewaySubnet
# (GatewaySubnet must already exist in hub-vnet)
gateway_subnet_id = (
    f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
    "/providers/Microsoft.Network/virtualNetworks/hub-vnet/subnets/GatewaySubnet"
)
pip_id = (
    f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
    "/providers/Microsoft.Network/publicIPAddresses/vpngw-pip"
)

gw_params = VirtualNetworkGateway(
    location=location,
    gateway_type="Vpn",
    vpn_type="RouteBased",
    sku=VirtualNetworkGatewaySku(name="VpnGw1", tier="VpnGw1"),
    enable_bgp=False,
    ip_configurations=[
        VirtualNetworkGatewayIPConfiguration(
            name="gwIpConfig",
            private_ip_allocation_method="Dynamic",
            subnet=SubResource(id=gateway_subnet_id),
            public_ip_address=SubResource(id=pip_id)
        )
    ]
)
# Gateway provisioning takes 20-45 minutes
poller = net.virtual_network_gateways.begin_create_or_update(resource_group, "hub-vpngw", gw_params)
print("VPN Gateway provisioning started — this takes 20-45 minutes")
gw = poller.result()
print(f"Gateway provisioned: {gw.name}  state: {gw.provisioning_state}")

In [ ]:
# Create Local Network Gateway (represents the on-premises VPN device)
lng_params = LocalNetworkGateway(
    location=location,
    gateway_ip_address="203.0.113.10",       # on-premises VPN device public IP
    local_network_address_space=AddressSpace(
        address_prefixes=["192.168.0.0/16"]  # on-premises network ranges
    )
)
poller = net.local_network_gateways.begin_create_or_update(resource_group, "onprem-lng", lng_params)
lng = poller.result()
print(f"Local network gateway: {lng.name}")

# Create the Site-to-Site connection
gw_id  = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Network/virtualNetworkGateways/hub-vpngw"
lng_id = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Network/localNetworkGateways/onprem-lng"

conn_params = VirtualNetworkGatewayConnection(
    location=location,
    connection_type="IPsec",
    virtual_network_gateway1=SubResource(id=gw_id),
    local_network_gateway2=SubResource(id=lng_id),
    shared_key="MySecretPresharedKey123!"  # must match on-premises VPN device
)
poller = net.virtual_network_gateway_connections.begin_create_or_update(
    resource_group, "hub-to-onprem", conn_params
)
conn = poller.result()
print(f"S2S connection: {conn.name}  state: {conn.connection_status}")

In [ ]:
# List VPN Gateway connections and their status
for conn in net.virtual_network_gateway_connections.list(resource_group):
    print(f"  {conn.name:<30} type: {conn.connection_type:<10} status: {conn.connection_status}")

# Check BGP peer status (if BGP is enabled)
gw_name = "hub-vpngw"
poller = net.virtual_network_gateways.begin_get_bgp_peer_status(resource_group, gw_name)
for peer in (poller.result().value or []):
    print(f"  BGP peer {peer.neighbor:<20} AS: {peer.asn}  state: {peer.state}")

## Summary

| Concept | Key Takeaway |
|---|---|
| VNet Peering | Low-latency backbone connection between VNets; non-transitive; no overlapping CIDRs |
| Regional vs Global peering | Same region vs cross-region; global peering has higher data transfer cost |
| Gateway transit | Hub enables transit so spoke peerings can use the hub's gateway to reach on-prem |
| Hub-and-spoke | Hub holds shared services (gateway, firewall, Bastion); spokes peer to hub |
| VPN Gateway | IPsec/IKEv2 tunnels over internet; S2S for branches, P2S for clients, V2V for VNets |
| GatewaySubnet | Required dedicated subnet for VPN/ExpressRoute gateways; minimum /27 |
| Active-active VPN | Two gateway instances; faster failover than active-standby |
| Local Network Gateway | Azure resource representing the on-premises VPN device and address space |
| ExpressRoute | Private dedicated circuit via partner; bypasses internet; consistent low latency |
| ExpressRoute Local/Standard/Premium | Local = single nearby region; Standard = geopolitical; Premium = global |
| Azure private peering | Connects ER circuit to Azure VNets |
| Microsoft peering | Connects ER circuit to Microsoft 365 and Azure PaaS public endpoints |
| ExpressRoute Global Reach | Links two ER circuits — site-to-site via Microsoft backbone |
| ExpressRoute FastPath | Bypasses ER gateway for data plane — lower latency, higher throughput |
| VPN + ER coexistence | ER as primary; VPN as failover — both connected to same VNet |
| Azure Virtual WAN | Managed hub-and-spoke at scale — any-to-any routing, automatic gateway management |
| VPN vs ExpressRoute | VPN = cheaper, quick setup, internet; ER = private, consistent, compliance |